# Libs

In [40]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from astropy.visualization import ZScaleInterval
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.utils.exceptions import AstropyWarning
from pathlib import Path
from typing import Tuple, Dict, Optional, Union
from matplotlib.lines import Line2D
import matplotlib.patches as patches
from scipy.ndimage import distance_transform_edt
import cv2
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import distance_transform_edt
from astropy.visualization import ZScaleInterval, AsinhStretch, ImageNormalize
import json
from pathlib import Path

# Plot params

In [4]:
# ====================================================================
# Publication-quality plot settings (ApJ, MNRAS, A&A standards)
# ====================================================================
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "Times New Roman"],
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.linewidth": 1.2,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

# Objects params

In [6]:
# Ваш словарь объектов
obj_dist_step_pc = {
    "Auriga": [450.0, 3, "Lada et al. 2009"],
    "L1495": [140.0, 3, "Galli et al. 2018 / Schlafly et al. 2014"],
    "L43_Karoly_2023": [125.0, 2, "Ortiz-León et al. 2017"],
    "MonR2": [830.0, 3, "Dzib et al. 2016"],
    "NGC1333": [299.0, 3, "Zucker et al. 2018"],
    "OMC1_450": [388.0, 3, "Kounkel et al. 2017"],
    "OMC1": [388.0, 3, "Kounkel et al. 2017"], # ± 5 pc
    "Perseus_B1": [293.0, 3, "Zucker et al. 2018"],
    "Rosette": [1330.0, 3, "Heyer et al. 2006"],
    "l1689b": [147.0, 3, "Ortiz-León et al. 2017"],
    "oph_a": [139.0, 3, "Zucker et al. 2019"],
    "oph_b": [139.0, 3, "Zucker et al. 2019"],
    "oph_c": [139.0, 3, "Zucker et al. 2019"]
}

# Funcs

In [42]:
class PolarizationAnalyzer:
    """
    Pipeline for analyzing dust polarization maps. 
    Focuses on generating structural maps and the polarization angle 
    structure function without assuming an underlying turbulence model.
    """
    def __init__(self, base_archive_path: Union[str, Path]):
        """
        Initializes the analyzer with the root directory of the archive.
        
        Args:
            base_archive_path (str or Path): Path to the BISTRO data archive.
            
        Raises:
            FileNotFoundError: If the provided directory does not exist.
        """
        self.base_dir = Path(base_archive_path)
        if not self.base_dir.exists():
            raise FileNotFoundError(f"Base directory not accessible: {self.base_dir}")
        
    def _load_fits(self, file_path: Path) -> Tuple[np.ndarray, Optional[WCS]]:
        """
        Loads FITS data and collapses degenerate axes.
        
        Args:
            file_path (Path): Path to the target FITS file.
            
        Returns:
            Tuple[np.ndarray, Optional[WCS]]: 2D image array and its celestial WCS.
            
        Raises:
            FileNotFoundError: If the FITS file is missing.
        """
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
            
        with fits.open(file_path) as hdul:
            data = np.squeeze(hdul[0].data)
            header = hdul[0].header
            
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', AstropyWarning)
                wcs = WCS(header) if 'CTYPE1' in header else None
                if wcs is not None and wcs.pixel_n_dim > 2:
                    wcs = wcs.celestial
        return data, wcs
    
    def _get_robust_limits(self, I: np.ndarray) -> Tuple[float, float]:
        """
        Safely computes display limits, falling back to percentiles if ZScale fails.
        """
        valid_I = I[np.isfinite(I)]
        if valid_I.size == 0:
            return 0.0, 1.0 # Fallback for completely empty maps
            
        try:
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(valid_I)
            # Ensure vmax is strictly greater than vmin
            if vmax <= vmin:
                raise ValueError("ZScale produced invalid limits.")
        except Exception:
            # Fallback to robust percentiles
            vmin, vmax = np.percentile(valid_I, [2, 98])
            
        # Absolute final safety check
        if vmax <= vmin:
            vmin = np.nanmin(valid_I)
            vmax = np.nanmax(valid_I)
            if vmax <= vmin: # Map is perfectly flat
                 vmax = vmin + 1.0 
                 
        return vmin, vmax
    
    def load_stokes_maps(self, object_name: str, band: str = "850um") -> Tuple[np.ndarray, np.ndarray, np.ndarray, WCS]:
        """
        Loads Stokes I, Q, and U maps for a specific target.
        
        The method first attempts to find files using the strict BISTRO naming convention:
        '{object_name}_{band}_{stokes}_map.fits'. 
        If strict files are not found, it falls back to a flexible search for any 
        FITS file containing '{stokes}_map' in its filename, accommodating both 
        '.fits' and '.fit' extensions.

        Args:
            object_name (str): Target directory name.
            band (str): Observation band (e.g., "850um").
            
        Returns:
            Tuple[np.ndarray, np.ndarray, np.ndarray, WCS]: Synchronized I, Q, U maps and WCS.
            
        Raises:
            FileNotFoundError: If any of the Stokes components cannot be located.
        """
        maps_dir = self.base_dir / object_name / "maps"
        if not maps_dir.exists():
            raise FileNotFoundError(f"Maps directory not found: {maps_dir}")

        stokes_data = {}
        for component in ['I', 'Q', 'U']:
            # 1. Primary strict search
            strict_name = f"{object_name}_{band}_{component}_map.fits"
            file_path = maps_dir / strict_name
            
            # 2. Fallback flexible search (now accepting .fit and .fits)
            if not file_path.exists():
                flexible_pattern = f"*{component}_map*.fit*"
                found_files = list(maps_dir.glob(flexible_pattern))
                
                if found_files:
                    file_path = found_files[0]  # Take the first matching file
                else:
                    raise FileNotFoundError(
                        f"Could not find Stokes {component} map for {object_name}. "
                        f"Tried strict '{strict_name}' and pattern '{flexible_pattern}'."
                    )
            
            data, wcs_loaded = self._load_fits(file_path)
            stokes_data[component] = data
            
            # Use WCS from the Stokes I map as the reference
            if component == 'I':
                wcs = wcs_loaded

        I, Q, U = stokes_data['I'], stokes_data['Q'], stokes_data['U']

        # Shape synchronization check to prevent broadcasting errors
        if Q.shape != I.shape or U.shape != I.shape:
            print(f"[Warning] Shape mismatch for {object_name}. Synchronizing Q/U to I: {I.shape}")
            ny, nx = I.shape
            Q = Q[:ny, :nx]
            U = U[:ny, :nx]
            
        return I, Q, U, wcs

    @staticmethod
    def compute_evpa(Q: np.ndarray, U: np.ndarray) -> np.ndarray:
        """
        Computes the Electric Vector Position Angle (EVPA).
        
        Args:
            Q (np.ndarray): Stokes Q map.
            U (np.ndarray): Stokes U map.
            
        Returns:
            np.ndarray: EVPA map in degrees (-90 to +90).
        """
        return np.degrees(0.5 * np.arctan2(U, Q))

    @staticmethod
    def pol_vector_components(p: Union[float, np.ndarray], psi_rad: Union[float, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
        """
        Calculates Cartesian vector components for plotting the B-field.
        Includes a +pi/2 rotation to convert EVPA to magnetic field orientation.
        
        Args:
            p (float or np.ndarray): Polarization degree or constant length.
            psi_rad (float or np.ndarray): EVPA in radians.
            
        Returns:
            Tuple[np.ndarray, np.ndarray]: X and Y vector components (vx, vy).
        """
        psi_shifted = psi_rad + np.pi / 2.0
        return p * np.cos(psi_shifted), p * np.sin(psi_shifted)

    @staticmethod
    def estimate_rms(data: np.ndarray) -> float:
        """
        Robustly estimates the Root Mean Square (RMS) noise using the Median Absolute Deviation (MAD).
        
        Args:
            data (np.ndarray): 2D map array (e.g., Stokes I or polarized intensity).
            
        Returns:
            float: Estimated background noise level. Returns 0.0 if map is empty/invalid.
        """
        valid = data[np.isfinite(data) & (data != 0)]
        if valid.size == 0: return 0.0
        return 1.4826 * np.median(np.abs(valid - np.median(valid)))

    def get_physical_scales(self, wcs: WCS, distance_pc: float, shape: Tuple[int, int]) -> Dict[str, float]:
        """
        Converts pixel grid scales to physical dimensions.
        
        Args:
            wcs (WCS): World Coordinate System object.
            distance_pc (float): Distance to the source in parsecs.
            shape (Tuple[int, int]): Dimensions of the image (Ny, Nx).
            
        Returns:
            Dict[str, float]: Dictionary containing pixel scales (arcsec, pc) and total field of view.
        """
        pix_scale_deg = np.mean(proj_plane_pixel_scales(wcs))
        pix_scale_arcsec = pix_scale_deg * 3600.0
        pix_scale_pc = np.radians(pix_scale_deg) * distance_pc
        
        return {
            "distance_pc": distance_pc,
            "pix_scale_arcsec": pix_scale_arcsec,
            "phys_scale_pc": pix_scale_pc,
            "angular_size_arcmin": (shape[0] * pix_scale_deg * 60, shape[1] * pix_scale_deg * 60),
            "physical_size_pc": (shape[0] * pix_scale_pc, shape[1] * pix_scale_pc),
            "total_pixels": shape[0] * shape[1]
        }

    @staticmethod
    def compute_structure_function(Q: np.ndarray, U: np.ndarray, R_list: np.ndarray, 
                                   min_pI: Optional[float] = None, bin_width: float = 0.5) -> Dict[str, np.ndarray]:
        """
        Computes the polarization angle structure function D_phi(R) = <sin^2(phi_1 - phi_2)>.
        
        Args:
            Q (np.ndarray): Stokes Q map.
            U (np.ndarray): Stokes U map.
            R_list (np.ndarray): Array of pixel lags (radii) to compute the function over.
            min_pI (Optional[float]): Minimum polarized intensity threshold. Defaults to None.
            bin_width (float): Tolerance window for pixel pair distance matching. Defaults to 0.5.
            
        Returns:
            Dict[str, np.ndarray]: Dictionary with keys 'R' (radii), 'Dphi' (structure function), 'Npairs' (counts).
        """
        Q, U = np.asarray(Q, dtype=float), np.asarray(U, dtype=float)
        Ny, Nx = Q.shape
        pI = np.hypot(Q, U)
        
        if min_pI is None:
            min_pI = -np.inf
            
        valid = np.isfinite(Q) & np.isfinite(U) & (pI > min_pI)
        q = np.where(valid, Q / pI, np.nan)
        u = np.where(valid, U / pI, np.nan)

        Dphi = np.full_like(R_list, np.nan, dtype=float)
        Npairs = np.zeros_like(R_list, dtype=int)

        def accumulate_shift(dy: int, dx: int) -> Optional[np.ndarray]:
            y1s, y2s = slice(max(0, dy), min(Ny, Ny + dy)), slice(max(0, -dy), min(Ny, Ny - dy))
            x1s, x2s = slice(max(0, dx), min(Nx, Nx + dx)), slice(max(0, -dx), min(Nx, Nx - dx))
            q1, u1 = q[y1s, x1s], u[y1s, x1s]
            q2, u2 = q[y2s, x2s], u[y2s, x2s]
            m = np.isfinite(q1) & np.isfinite(q2)
            if not np.any(m): return None
            dq, du = q1[m] - q2[m], u1[m] - u2[m]
            return 0.25 * (dq**2 + du**2)

        for i, R in enumerate(R_list):
            if R < 0: continue
            rmax = int(np.ceil(R + bin_width))
            shifts = [(dy, dx) for dy in range(-rmax, rmax + 1) for dx in range(-rmax, rmax + 1) 
                      if (dx != 0 or dy != 0) and abs(np.hypot(dx, dy) - R) <= bin_width]

            contrib_list = [accumulate_shift(dy, dx) for dy, dx in shifts]
            valid_contribs = [c for c in contrib_list if c is not None and c.size > 0]
            if valid_contribs:
                contrib_vec = np.concatenate(valid_contribs)
                Dphi[i] = np.nanmean(contrib_vec)
                Npairs[i] = contrib_vec.size

        return {"R": R_list, "Dphi": Dphi, "Npairs": Npairs}

    def plot_polarization_map(self, I: np.ndarray, vx: np.ndarray, vy: np.ndarray, wcs: WCS, object_name: str, step: int = 4, save_path: str = None):
        """
        Plots the classic dust intensity map overlaid with magnetic field vectors.
        
        Args:
            I (np.ndarray): Stokes I map (background).
            vx (np.ndarray): B-field vector X component.
            vy (np.ndarray): B-field vector Y component.
            wcs (WCS): World Coordinate System for projection.
            object_name (str): Target name for the title.
            step (int): Decimation factor for the vector grid.
        """
        interval = ZScaleInterval()
        vmin, vmax = interval.get_limits(I)
        
        x = np.linspace(0, vx.shape[1] - 1, vx.shape[1])
        y = np.linspace(0, vx.shape[0] - 1, vx.shape[0])
        px, py = np.meshgrid(x, y)

        jcmtmap_mask = np.ones(I.shape)
        #jcmtmap_mask[I < vmax * 0.6] = np.nan

        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection=wcs)
        im = ax.imshow(I, origin="lower", vmin=vmin, vmax=vmax, cmap='viridis')
        
        ax.quiver(
            px[::step, ::step] * jcmtmap_mask[::step, ::step],
            py[::step, ::step] * jcmtmap_mask[::step, ::step],
            vx[::step, ::step] * jcmtmap_mask[::step, ::step],
            vy[::step, ::step] * jcmtmap_mask[::step, ::step],
            angles='xy', scale_units='xy', scale=0.2,
            headwidth=0, headlength=0, headaxislength=0,
            pivot="mid", width=0.003, color="k"
        )

        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")
        ax.set_xlabel("RA (J2000)")
        ax.set_ylabel("Dec (J2000)")
        ax.set_title(rf"{object_name} -- Magnetic Field Morphology")
        if save_path:
            fig.savefig(save_path, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()
        
    def plot_snr_colored_polarization_map(self, I: np.ndarray, snr_map: np.ndarray, 
                                          vx: np.ndarray, vy: np.ndarray, wcs: WCS, 
                                          object_name: str, step: int = 4, save_path: str = None):
        """
        Plots a B-field map where vectors are colored by their SNR levels:
        1 < SNR <= 3 (cyan), 3 < SNR <= 5 (red), SNR > 5 (magenta).
        """
        vmin, vmax = self._get_robust_limits(I)
        
        x = np.linspace(0, vx.shape[1] - 1, vx.shape[1])
        y = np.linspace(0, vx.shape[0] - 1, vx.shape[0])
        px, py = np.meshgrid(x, y)

        intensity_mask = np.ones(I.shape)
        #intensity_mask[I < vmax * 0.6] = np.nan

        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection=wcs)
        im = ax.imshow(I, origin="lower", vmin=vmin, vmax=vmax, cmap='viridis')

        # Цвета: бирюзовый вместо оранжевого
        levels = [2, 3, 5]
        colors = ['magenta', 'red', 'cyan']
        # Понятные математические подписи
        labels = [fr'{levels[0]} $<$ SNR $\le$ {levels[1]}', fr'{levels[1]} $<$ SNR $\le$ {levels[2]}', fr'SNR $>$ {levels[2]}']

        for i in range(len(levels)):
            lower = levels[i]
            upper = levels[i+1] if i+1 < len(levels) else np.inf
            
            snr_mask = np.where((snr_map > lower) & (snr_map <= upper), 1.0, np.nan)
            combined_mask = intensity_mask * snr_mask
            
            ax.quiver(
                px[::step, ::step] * combined_mask[::step, ::step],
                py[::step, ::step] * combined_mask[::step, ::step],
                vx[::step, ::step] * combined_mask[::step, ::step],
                vy[::step, ::step] * combined_mask[::step, ::step],
                angles='xy', scale_units='xy', scale=0.2,
                headwidth=0, headlength=0, headaxislength=0,
                pivot="mid", width=0.004, color=colors[i]
            )

        # Легенда с использованием прокси-артистов
        legend_elements = [Line2D([0], [0], color=c, lw=2, label=l) for c, l in zip(colors, labels)]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=9, framealpha=0.8, edgecolor='black')

        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")
        ax.set_xlabel("RA (J2000)")
        ax.set_ylabel("Dec (J2000)")
        ax.set_title(rf"{object_name.replace('_', ' ')} -- SNR-Coded B-Field")
        
        if save_path:
            fig.savefig(save_path, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()
            
    def plot_structure_function(self, sf_data: Dict[str, np.ndarray], save_path: str = None):
            """
            Plots the raw and binned polarization angle structure function.
            """
            R, D, N = sf_data["R_pc"], sf_data["Dphi"], sf_data["Npairs"]
            m = np.isfinite(D) & (N > 0) & (R > 0)
            Rv, Dv, Nv = R[m], D[m], N[m]

            #m2 = np.isfinite(Dv) & (Dv > 0)
            #Rv, Dv, Nv = Rv[m2], Dv[m2], Nv[m2]
            
            if len(Rv) == 0:
                print(f"[Warning] No valid data for structure function plot. Skipping...")
                return
            
            sig = Dv / np.sqrt(Nv)

            def logbin_median(x, y, nbins=12):
                lx = np.log10(x)
                edges = np.linspace(lx.min(), lx.max(), nbins + 1)
                xb, yb = [], []
                for a, b in zip(edges[:-1], edges[1:]):
                    sel = (lx >= a) & (lx < b)
                    if np.any(sel):
                        xb.append(10**np.median(lx[sel]))
                        yb.append(np.median(y[sel]))
                return np.array(xb), np.array(yb)

            Rb_s, Db_s = logbin_median(Rv, Dv, nbins=10)

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.plot(Rv, Dv, marker="o", ms=4, lw=1, alpha=0.6, color='gray', label=r"Raw $D_\phi(R)$")
            ax.errorbar(Rv, Dv, yerr=sig, fmt="none", ecolor='gray', lw=0.8, alpha=0.5)

            if len(Rb_s) > 1:
                ax.plot(Rb_s, Db_s, marker="s", ms=6, lw=2, color='black', label=r"Log-binned Median")

            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel(r"Separation $R$ (pc)")
            ax.set_ylabel(r"Structure Function $D_\phi(R)$ (rad$^2$)")
            ax.set_title(r"Polarization Angle Dispersion")

            ax.minorticks_on()
            ax.grid(True, which="major", lw=0.8, alpha=0.3)
            ax.grid(True, which="minor", lw=0.4, alpha=0.15)
            ax.legend(frameon=True, edgecolor='black')

            if save_path:
                fig.savefig(save_path, bbox_inches='tight')
                plt.close(fig)
            else:
                plt.show()
            
            
    def plot_snr_distributions(self, p_frac: np.ndarray, psi_deg: np.ndarray, snr_pI: np.ndarray, object_name: str, save_path: Optional[str] = None):
        """
        Plots histograms of Polarization Fraction and EVPA for different 
        SNR thresholds (>1, >3, >5). Essential for analyzing Ricean bias 
        and the statistical ordering of the magnetic field.
        """
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        
        thresholds = [2, 3, 5]
        colors = ['gray', 'royalblue', 'crimson']
        alphas = [0.4, 0.7, 0.9]
        
        for thr, col, alp in zip(thresholds, colors, alphas):
            mask = (snr_pI > thr) & np.isfinite(p_frac) & np.isfinite(psi_deg)
            valid_p = p_frac[mask]
            valid_psi = psi_deg[mask]
            
            if len(valid_p) == 0:
                continue
                
            # 1. Polarization Fraction Distribution
            bins_p = np.linspace(0, 0.3, 40) # Capped at 30% for visibility
            ax1.hist(valid_p, bins=bins_p, histtype='step', lw=2, 
                     color=col, alpha=alp, density=True, 
                     label=rf'$\sigma > {thr}$ ($N={len(valid_p)}$)')
            
            # 2. EVPA Distribution
            bins_psi = np.linspace(-90, 90, 36)
            ax2.hist(valid_psi, bins=bins_psi, histtype='step', lw=2, 
                     color=col, alpha=alp, density=True, 
                     label=rf'$\sigma > {thr}$')

        ax1.set_xlabel(r"Polarization Fraction $p$")
        ax1.set_ylabel("Probability Density")
        ax1.legend(frameon=False)
        ax1.grid(True, which="major", alpha=0.3)

        ax2.set_xlabel(r"EVPA $\psi$ (deg)")
        ax2.set_ylabel("Probability Density")
        ax2.legend(frameon=False)
        ax2.grid(True, which="major", alpha=0.3)

        fig.suptitle(rf"{object_name.replace('_', ' ')} -- Distributions by Polarized SNR")
        fig.tight_layout()
        
        if save_path:
            for fmt in ['pdf', 'png']:
                fig.savefig(f"{save_path}.{fmt}", bbox_inches='tight')
                plt.close(fig)
        else:
            plt.show()

    
    def detect_hills_statistically(self, I, wcs, object_name, var_I=None, qual_I=None, margin_ratio=0.15, sigma_level=2.0, save_path=None):
        import cv2
        import numpy as np
        import matplotlib.pyplot as plt
        from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch
        from scipy.ndimage import distance_transform_edt
        from astropy.stats import sigma_clipped_stats

        ny, nx = I.shape
        
        # --- 1. ГЕОМЕТРИЯ (Ваш "забор") ---
        footprint = np.isfinite(I) & (I != 0)
        dist_map = distance_transform_edt(footprint)
        threshold_dist = np.nanmax(dist_map) * margin_ratio
        safe_zone_mask = dist_map >= threshold_dist

        # --- 2. ЛОГИКА МАСКИРОВАНИЯ ---
        has_quality = (qual_I is not None)
        
        if has_quality:
            # РЕЖИМ А: ПАСПОРТ КАЧЕСТВА (Игнорируем sigma_level)
            # Берем только "хорошие" пиксели (значение 0) внутри забора
            cloud_mask = (qual_I == 0) & safe_zone_mask
            
            method_str = "Hardware QUALITY Mask"
            title_str = fr"{object_name}: clumps (via QUALITY flags)"
            
        else:
            # РЕЖИМ Б: СТАТИСТИЧЕСКАЯ ЭВРИСТИКА (Ваш алгоритм с sigma_level)
            data_in_zone = I[safe_zone_mask & np.isfinite(I)]
            
            if data_in_zone.size == 0:
                print(f"[WARN] Safe zone is empty for {object_name}")
                return [], np.full_like(I, np.inf), np.zeros_like(I), {}

            median_val = np.nanmedian(data_in_zone)
            std_val = np.nanstd(data_in_zone)
            
            # Порог: медиана + N сигм
            science_threshold = median_val + sigma_level * std_val
            cloud_mask = (I > science_threshold) & safe_zone_mask
            
            method_str = f"{sigma_level}$\\sigma$ Heuristic"
            title_str = fr"{object_name}: clumps ($>${sigma_level}$\sigma$ above median)"

        # --- 3. ПОИСК КОНТУРОВ (OpenCV) ---
        cloud_mask_uint8 = cloud_mask.astype('uint8') * 255
        contours, _ = cv2.findContours(cloud_mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Отсеиваем мелкий мусор сразу, чтобы не тащить его в физику
        valid_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > 12]

        # --- 4. РАСЧЕТ ОШИБОК И ФОНА ---
        # Создаем маску найденных облаков, чтобы вырезать их из фона
        exclusion_mask = np.zeros_like(cloud_mask_uint8)
        for cnt in valid_contours:
            cv2.drawContours(exclusion_mask, [cnt], -1, 255, thickness=cv2.FILLED)
            
        # Честный фон: внутри забора, но вне облаков
        bg_mask = safe_zone_mask & (exclusion_mask == 0)
        bg_data = I[bg_mask & np.isfinite(I)]
        
        if bg_data.size > 0:
            mean_bg, median_bg, std_bg = sigma_clipped_stats(bg_data, sigma=3.0)
        else:
            mean_bg, median_bg, std_bg = np.nan, np.nan, np.nanstd(I)

        # Формируем карту ошибок для физики векторов
        if var_I is not None:
            # Если есть VARIANCE, используем его для SNR векторов (даже если нет QUALITY)
            err_I_map = np.sqrt(np.where(var_I > 0, var_I, np.inf))
            if has_quality:
                err_I_map[qual_I != 0] = np.inf
            snr_map = np.where(err_I_map < np.inf, I / err_I_map, 0)
            if not has_quality:
                method_str += " + Hardware Variance"
        else:
            # Если аппаратных ошибок вообще нет, полагаемся на фон
            err_I_map = np.full_like(I, std_bg)
            err_I_map[~np.isfinite(I)] = np.inf
            snr_map = np.where(std_bg > 0, I / std_bg, 0)

        # --- 5. ОТРИСОВКА ---
        fig = plt.figure(figsize=(8, 7))
        ax = fig.add_subplot(111, projection=wcs)
        
        norm = ImageNormalize(I, interval=ZScaleInterval(), stretch=AsinhStretch(a=0.1))
        im = ax.imshow(I, origin='lower', cmap='viridis', norm=norm)

        # Рисуем забор пунктиром
        ax.contour(safe_zone_mask, levels=[0.5], colors='magenta', linewidths=1.2, alpha=0.8, linestyles='dashed')

        # Рисуем финальные контуры
        count = len(valid_contours)
        for cnt in valid_contours:
            poly = cnt.reshape(-1, 2)
            poly = np.vstack([poly, poly[0]])
            ax.plot(poly[:, 0], poly[:, 1], color='red', lw=1.5)

        # Обновляем заголовок с актуальным количеством объектов
        final_title = title_str.replace("clumps", f"{count} clumps")
        ax.set_title(final_title)
        
        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")

        if save_path:
            plt.savefig(str(save_path), bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()

        stats_dict = {'mean_bg': mean_bg, 'median_bg': median_bg, 'std_bg': std_bg, 'method': method_str}
        return valid_contours, err_I_map, snr_map, stats_dict
    
    
    def load_stokes_maps_with_variance(self, object_name: str, band: str = "850um"):
        """
        Гибкий поиск файлов: ищет вхождения (object_name, parameter, 'map') 
        в любом регистре и порядке.
        """
        from astropy.io import fits
        from astropy.wcs import WCS
        import numpy as np
        import os

        maps_dir = self.base_dir / object_name / "maps"
        if not maps_dir.exists():
            raise FileNotFoundError(f"Директория не найдена: {maps_dir}")

        # Подготовка "отпечатка" объекта: нижний регистр, без дефисов и подчеркиваний
        clean_obj = object_name.lower().replace('_', '').replace('-', '')
        all_files = [f for f in os.listdir(maps_dir) if f.lower().endswith(('.fits', '.fit'))]

        def find_file(stokes_char):
            s_low = stokes_char.lower()
            # Защита от поиска одинокой буквы внутри других слов (например, 'i' в 'auriga')
            patterns = [f"_{s_low}_", f"{s_low}map", f"{s_low}_map"]
            
            for f in all_files:
                f_low = f.lower()
                # Тотальная зачистка имени файла для безопасного сравнения объекта
                f_clean_name = f_low.replace('_', '').replace('-', '')
                
                is_obj = clean_obj in f_clean_name
                has_stokes = any(p in f_low for p in patterns)
                
                if is_obj and has_stokes and "map" in f_low:
                    return maps_dir / f
            return None

        # Ищем пути для I, Q, U
        found_paths = {p: find_file(p) for p in ['I', 'Q', 'U']}

        # Проверка на вылет
        for p, path in found_paths.items():
            if path is None:
                print(f"[ERROR] Не удалось найти файл для параметра {p} в {maps_dir}")
                print(f"Искали: {clean_obj} + {p.lower()} + 'map'")
                raise FileNotFoundError(f"Missing {p} map for {object_name}")

        def extract_fits_layers(filepath):
            with fits.open(filepath) as hdul:
                # HDU 0: Основные данные
                data = np.squeeze(hdul[0].data)
                # HDU 1: VARIANCE (если есть)
                var_data = np.squeeze(hdul[1].data) if len(hdul) > 1 else None
                # HDU 2: QUALITY (если есть)
                qual_data = np.squeeze(hdul[2].data) if len(hdul) > 2 else None
                return data, var_data, qual_data

        # Извлечение данных с использованием найденных путей
        I, var_I, qual_I = extract_fits_layers(found_paths['I'])
        Q, var_Q, qual_Q = extract_fits_layers(found_paths['Q'])
        U, var_U, qual_U = extract_fits_layers(found_paths['U'])

        # Чтение WCS из найденного файла I
        with fits.open(found_paths['I']) as hdul_I:
            wcs = WCS(hdul_I[0].header).celestial 

        return I, Q, U, wcs, var_I, var_Q, var_U, qual_I, qual_Q, qual_U
    
    
    def process_object(self, object_name: str, distance_pc: float, band: str = "850um", margin_ratio=0.08, sigma_level=1.5, step: int = 2):

        print(f"\n{'='*60}\nPROCESSING: {object_name}\n{'='*60}")

        try:
            # ОЖИДАЕМАЯ СИГНАТУРА ЗАГРУЗКИ:
            # Если var_* или qual_* файлов нет, функция загрузки должна возвращать None на их местах.
            I, Q, U, wcs, var_I, var_Q, var_U, qual_I, qual_Q, qual_U = self.load_stokes_maps_with_variance(object_name, band)
        except FileNotFoundError:
            return None, None, None

        report_dir = self.base_dir.parents[1] / "report"
        plots_dir = report_dir / "plots" / object_name
        plots_dir.mkdir(parents=True, exist_ok=True)

        # --- ЭТАП 1: СЕГМЕНТАЦИЯ СТРУКТУР ---
        diag_path = plots_dir / f"{object_name}_stat_segmentation.pdf"
        contours, err_I_map, snr_I_map, stats_I = self.detect_hills_statistically(
            I, wcs, object_name, 
            var_I=var_I, qual_I=qual_I,
            margin_ratio=margin_ratio, 
            sigma_level=sigma_level, 
            save_path=str(diag_path)
        )

        if not contours:
            print(f"[ERROR] No clumps found for {object_name}. Physics is canceled.")
            return None, None, None

        # --- ЭТАП 2: ФИЗИЧЕСКАЯ ИЗОЛЯЦИЯ ---
        science_mask = np.zeros(I.shape, dtype=np.uint8)
        for cnt in contours:
            if cv2.contourArea(cnt) > 12: 
                cv2.drawContours(science_mask, [cnt], -1, 1, thickness=cv2.FILLED)

        # Зачистка битых пикселей I
        if qual_I is not None:
            science_mask[qual_I != 0] = 0

        science_mask_bool = science_mask.astype(bool)

        I_full = I.copy()
        I[~science_mask_bool] = np.nan
        Q[~science_mask_bool] = np.nan
        U[~science_mask_bool] = np.nan

        # --- ЭТАП 3: СТРОГАЯ ПРОПАГАЦИЯ ОШИБОК И ФИЗИКА ---
        def get_stokes_error_map(data, var_map, qual_map, science_mask_bool):
            """Внутренний Fallback для карт Q и U"""
            # ИСПРАВЛЕНИЕ: Проверяем только var_map
            if var_map is not None:
                err = np.sqrt(np.where(var_map > 0, var_map, np.inf))
                if qual_map is not None:
                    err[qual_map != 0] = np.inf
                return err
            else:
                # Эмпирический расчет по фону
                bg_data = data[~science_mask_bool & np.isfinite(data)]
                from astropy.stats import sigma_clipped_stats
                std_val = sigma_clipped_stats(bg_data, sigma=3.0)[2] if bg_data.size > 0 else np.nanstd(data)
                return np.full_like(data, std_val)

        # 1. Извлекаем ошибки Q и U
        err_Q = get_stokes_error_map(Q, var_Q, qual_Q, science_mask_bool)
        err_U = get_stokes_error_map(U, var_U, qual_U, science_mask_bool)

        # 2. Интенсивность поляризации (Сырая)
        pI_raw = np.hypot(Q, U)

        # 3. Ошибка интенсивности поляризации (Аналитическая)
        pI_safe = np.where(pI_raw > 0, pI_raw, np.inf)
        noise_pI = np.sqrt(Q**2 * err_Q**2 + U**2 * err_U**2) / pI_safe

        # 4. ДЕБИАСИНГ (Коррекция Уордла-Кронберга)
        radicand = pI_raw**2 - noise_pI**2
        pI_debiased = np.sqrt(np.maximum(radicand, 0)) # np.maximum изящнее чем np.where

        # 5. Итоговый SNR для векторов (по дебиасированному PI)
        snr_pI_map = np.where(noise_pI > np.finfo(float).eps, pI_debiased / noise_pI, 0)

        # 6. Жесткий фильтр для структурных функций (оставляем только достоверные векторы)
        # Пиксели с SNR_PI < 3 станут NaN
        valid_vectors = (snr_pI_map >= 2.0)
        Q[~valid_vectors] = np.nan
        U[~valid_vectors] = np.nan

        # Ошибка угла (для взвешивания в структурных функциях, опционально)
        # noise_psi_rad = (1.0 / (2.0 * pI_safe**2)) * np.sqrt(Q**2 * err_U**2 + U**2 * err_Q**2)

        psi_deg = self.compute_evpa(Q, U)
        vx, vy = self.pol_vector_components(p=1.0, psi_rad=np.radians(psi_deg))
        scales = self.get_physical_scales(wcs, distance_pc, I_full.shape)

    # --- ЭТАП 4: ГРАФИКА И ВЫВОД ---
        ny, nx = I.shape
        max_R = int(min(ny, nx) * (2 / 3))
        R_list_px = np.arange(1, max_R + 1)

        # Расчет структурной функции
        sf_data = self.compute_structure_function(Q, U, R_list=R_list_px, min_pI=None)

        if sf_data is not None:
            sf_data["R_pc"] = R_list_px * scales["phys_scale_pc"]
            self.plot_structure_function(sf_data, save_path=plots_dir/f"{object_name}_sf.pdf")

        # Отрисовка карт
        self.plot_polarization_map(I_full, vx, vy, wcs, f"{object_name} (Full Context)", step, 
                                   save_path=plots_dir/f"{object_name}_pol_map_full.pdf")
        self.plot_polarization_map(I, vx, vy, wcs, f"{object_name} (Isolated Cores)", step, 
                                   save_path=plots_dir/f"{object_name}_pol_map_cropped.pdf")
        self.plot_snr_colored_polarization_map(I_full, snr_pI_map, vx, vy, wcs, f"{object_name} (Full Context)", step, 
                                               save_path=plots_dir/f"{object_name}_snr_colored_full.pdf")
        self.plot_snr_colored_polarization_map(I, snr_pI_map, vx, vy, wcs, f"{object_name} (Isolated Cores)", step, 
                                               save_path=plots_dir/f"{object_name}_snr_colored_cropped.pdf")

        # --- ЭТАП 5: ГЕНЕРАЦИЯ ОТЧЕТА ---
        valid_count = np.sum(valid_vectors & science_mask_bool)
        median_noise_I = np.nanmedian(err_I_map[science_mask_bool])

        # Вычисляем физический предел R для отчета
        r_min_pc = 1 * scales['phys_scale_pc']
        r_max_pc = max_R * scales['phys_scale_pc']

        # Определяем, какой метод использовался для Q и U
        method_QU = "Hardware Variance" if (var_Q is not None and var_U is not None) else f"Heuristic (Background RMS)"
        median_snr_pi = np.nanmedian(snr_pI_map[valid_vectors & science_mask_bool]) if valid_count > 0 else 0

        report = f"""
            ============================================================
            [+] OBJECT PASSPORT: {object_name}
            ============================================================
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    {stats_I['method']}
              - Stokes vectors Q/U:   {method_QU}
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     {distance_pc} pc
              - Lag range (R):        from 1 to {max_R} pixels
                                      ({r_min_pc:.4f} pc — {r_max_pc:.4f} pc)
              - Pixel scale:          {scales['phys_scale_pc']:.5f} pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       {valid_count} pixels (SNR_PI > 2.0)
              - Median noise (I):     {median_noise_I:.4f} mJy/beam
              - Median SNR (PI):      {median_snr_pi:.2f}
            ============================================================
            """
        print(report)

        # Упаковываем данные для возврата, если они нужны внешним скриптам
        stats = {
            "Distance (pc)": distance_pc,
            "R_range_px": (1, max_R),
            "R_range_pc": (r_min_pc, r_max_pc),
            "Effective Pixels": valid_count,
            "Method I": stats_I['method'],
            "Method QU": method_QU
        }
        # --- ЭТАП 6: СОХРАНЕНИЕ СТАТИСТИКИ ---
        # Формируем путь: report/plots/ObjectName/ObjectName_stats.json
        stats_path = plots_dir / f"{object_name}_stats.json"
        
        # Подготавливаем словарь (убеждаемся, что все типы данных сериализуемы)
        serializable_stats = {}
        for k, v in stats.items():
            if isinstance(v, (np.ndarray, np.generic)):
                serializable_stats[k] = v.tolist()
            else:
                serializable_stats[k] = v

        with open(stats_path, "w", encoding="utf-8") as f:
            json.dump(serializable_stats, f, indent=4, ensure_ascii=False)
            
        print(f"[DISK] Statistics saved to {stats_path}")

        return psi_deg, sf_data, stats

# Implemetation for one objrct

In [44]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "Auriga"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.08,
    sigma_level=0.5,
    step = step,

)


PROCESSING: Auriga


Set OBSGEO-B to    19.822828 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: Auriga
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    Hardware QUALITY Mask
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     450.0 pc
              - Lag range (R):        from 1 to 180 pixels
                                      (0.0087 pc — 1.5708 pc)
              - Pixel scale:          0.00873 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       1502 pixels (SNR_PI > 2.0)
              - Median noise (I):     3.9265 mJy/beam
              - Median SNR (PI):      2.71
            
[DISK] Statistics saved to report/plots/Auriga/Auriga_stats.json


In [46]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "L1495"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.2,
    sigma_level=0.75,
    step = step,

)


PROCESSING: L1495


Set OBSGEO-B to    19.822855 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: L1495
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    Hardware QUALITY Mask
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     140.0 pc
              - Lag range (R):        from 1 to 91 pixels
                                      (0.0054 pc — 0.4941 pc)
              - Pixel scale:          0.00543 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       78 pixels (SNR_PI > 2.0)
              - Median noise (I):     2.0814 mJy/beam
              - Median SNR (PI):      2.68
            
[DISK] Statistics saved to report/plots/L1495/L1495_stats.json


In [47]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "L43_Karoly_2023"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.2,
    sigma_level=0.3,
    step = step,

)


PROCESSING: L43_Karoly_2023


Set OBSGEO-B to    19.822867 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: L43_Karoly_2023
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    Hardware QUALITY Mask
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     125.0 pc
              - Lag range (R):        from 1 to 88 pixels
                                      (0.0048 pc — 0.4266 pc)
              - Pixel scale:          0.00485 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       189 pixels (SNR_PI > 2.0)
              - Median noise (I):     1.7596 mJy/beam
              - Median SNR (PI):      2.75
            
[DISK] Statistics saved to report/plots/L43_Karoly_2023/L43_Karoly_2023_stats.json


In [48]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "MonR2"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.15,
    sigma_level=0.1,
    step = step,

)


PROCESSING: MonR2


Set OBSGEO-B to    19.822828 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: MonR2
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    Hardware QUALITY Mask
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     830.0 pc
              - Lag range (R):        from 1 to 176 pixels
                                      (0.0161 pc — 2.8329 pc)
              - Pixel scale:          0.01610 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       4402 pixels (SNR_PI > 2.0)
              - Median noise (I):     0.0000 mJy/beam
              - Median SNR (PI):      3.79
            
[DISK] Statistics saved to report/plots/MonR2/MonR2_stats.json


CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "NGC1333"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.08,
    sigma_level=0.5,
    step = 3,

)

CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "OMC1_450"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.08,
    sigma_level=0.1,
    step = step,

)

In [51]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "OMC1"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.08,
    sigma_level=0.1,
    step = step,

)


PROCESSING: OMC1


Set OBSGEO-B to    19.822855 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: OMC1
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    0.1$\sigma$ Heuristic + Hardware Variance
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     388.0 pc
              - Lag range (R):        from 1 to 186 pixels
                                      (0.0075 pc — 1.3995 pc)
              - Pixel scale:          0.00752 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       6779 pixels (SNR_PI > 2.0)
              - Median noise (I):     0.0000 mJy/beam
              - Median SNR (PI):      9.08
            
[DISK] Statistics saved to report/plots/OMC1/OMC1_stats.json


In [52]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "Perseus_B1"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.15,
    sigma_level=0.3,
    step = step,

)


PROCESSING: Perseus_B1


Set OBSGEO-B to    19.822822 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: Perseus_B1
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    0.3$\sigma$ Heuristic + Hardware Variance
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     293.0 pc
              - Lag range (R):        from 1 to 60 pixels
                                      (0.0170 pc — 1.0228 pc)
              - Pixel scale:          0.01705 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       401 pixels (SNR_PI > 2.0)
              - Median noise (I):     0.0000 mJy/beam
              - Median SNR (PI):      3.87
            
[DISK] Statistics saved to report/plots/Perseus_B1/Perseus_B1_stats.json


In [53]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "Rosette"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.2,
    sigma_level=0.5,
    step = step,

)


PROCESSING: Rosette


Set OBSGEO-B to    19.822864 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: Rosette
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    Hardware QUALITY Mask
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     1330.0 pc
              - Lag range (R):        from 1 to 190 pixels
                                      (0.0258 pc — 4.9005 pc)
              - Pixel scale:          0.02579 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       1452 pixels (SNR_PI > 2.0)
              - Median noise (I):     3.2960 mJy/beam
              - Median SNR (PI):      2.56
            
[DISK] Statistics saved to report/plots/Rosette/Rosette_stats.json


In [54]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "oph_a"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.12,
    sigma_level=0.1,
    step = step,

)


PROCESSING: oph_a


Set OBSGEO-B to    19.822880 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: oph_a
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    Hardware QUALITY Mask
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     139.0 pc
              - Lag range (R):        from 1 to 176 pixels
                                      (0.0027 pc — 0.4744 pc)
              - Pixel scale:          0.00270 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       2288 pixels (SNR_PI > 2.0)
              - Median noise (I):     4.2117 mJy/beam
              - Median SNR (PI):      3.31
            
[DISK] Statistics saved to report/plots/oph_a/oph_a_stats.json


In [55]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "oph_b"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.08,
    sigma_level=0.5,
    step = step,

)


PROCESSING: oph_b


Set OBSGEO-B to    19.822880 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: oph_b
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    Hardware QUALITY Mask
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     139.0 pc
              - Lag range (R):        from 1 to 176 pixels
                                      (0.0027 pc — 0.4744 pc)
              - Pixel scale:          0.00270 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       1370 pixels (SNR_PI > 2.0)
              - Median noise (I):     4.1816 mJy/beam
              - Median SNR (PI):      2.68
            
[DISK] Statistics saved to report/plots/oph_b/oph_b_stats.json


In [56]:
CLEAN_ARCHIVE_PATH = "data/BISTRO_Clean"
analyzer = PolarizationAnalyzer(base_archive_path=CLEAN_ARCHIVE_PATH)

object_name = "oph_c"
object_dist = obj_dist_step_pc[object_name][0]
step = obj_dist_step_pc[object_name][1]

evpa_map, sf_results, stats = analyzer.process_object(
    object_name=object_name, 
    distance_pc=object_dist, 
    band="850um",
    margin_ratio=0.12,
    sigma_level=0.7,
    step = step,

)


PROCESSING: oph_c


Set OBSGEO-B to    19.822914 from OBSGEO-[XYZ].
Set OBSGEO-H to     4120.022 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]



            [+] OBJECT PASSPORT: oph_c
            > NOISE ESTIMATION METHODOLOGY
              - Stokes parameter I:    Hardware QUALITY Mask
              - Stokes vectors Q/U:   Hardware Variance
              - PI debiasing:         Applied (Wardle & Kronberg 1974)

            > GEOMETRY AND SCALES
              - Adopted distance:     139.0 pc
              - Lag range (R):        from 1 to 176 pixels
                                      (0.0027 pc — 0.4744 pc)
              - Pixel scale:          0.00270 pc/pixel

            > STATISTICS AND PHYSICAL SAMPLE
              - Effective area:       921 pixels (SNR_PI > 2.0)
              - Median noise (I):     4.7166 mJy/beam
              - Median SNR (PI):      2.69
            
[DISK] Statistics saved to report/plots/oph_c/oph_c_stats.json


# Implementation for all objs in dir with report in .tex

In [61]:
# Настройки путей
PLOTS_DIR = Path("report/plots") 
REPORT_DIR = Path("report/report_text")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Обновленная методология с учетом "цифровой археологии" и STARLINK
METHODOLOGY_TEXT = r"""
\section*{Methodology and Data Integrity}
This report provides a quantitative assessment of molecular cloud structures based on BISTRO survey data. A key feature of the current pipeline is the prioritized use of hardware-derived metadata for error propagation.

\paragraph{Noise Analysis Strategy:}
The pipeline implements a "Graceful Downgrade" logic. If the FITS headers and HDU structure (derived from the STARLINK \texttt{pol2map} reduction) contain \texttt{VARIANCE} and \texttt{QUALITY} extensions, these are utilized as the primary source of per-pixel uncertainty. This accounts for the non-uniform noise distribution inherent in the JCMT \textit{Daisy Scan} pattern. In the absence of such layers, the system reverts to a heuristic estimation of the RMS within a geometric safety zone, utilizing iterative sigma-clipping to isolate the background from the emission.

\paragraph{Debiasing and Physics:}
To counteract the systematic overestimation of polarized intensity in low SNR regimes (Ricean bias), we apply the \textbf{Wardle \& Kronberg (1974)} debiasing technique: $PI_{db} = \sqrt{PI^2 - \sigma_{PI}^2}$. Pixels failing the $SNR_{PI} > 2$ criterion are excluded from the structure function analysis to prevent noise-driven artifacts in the magnetic field morphology.
"""

latex_content = r"""\documentclass[10pt, a4paper]{article}
\usepackage[margin=1.5cm]{geometry}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{subcaption}
\usepackage{float}
\usepackage{amsmath}
\usepackage{mathpazo}
\usepackage[utf8]{inputenc}
\usepackage[T2A]{fontenc}
\usepackage[english,russian]{babel}
\usepackage{xcolor}

\title{BISTRO Polarimetric Report: Statistical Data Passport}
\author{Automated Analysis Pipeline}
\date{\today}

\begin{document}
\maketitle
""" + METHODOLOGY_TEXT + r"\newpage"

# Предположим, что словарь объектов определен выше
for object_name, params in obj_dist_step_pc.items():
    object_dist, step, ref = params
    obj_path = PLOTS_DIR / object_name
    
    if not obj_path.exists():
        continue

    f_seg = f"{object_name}_stat_segmentation.pdf"
    f_snr = f"{object_name}_snr_colored_full.pdf"
    f_sf = f"{object_name}_sf.pdf"
    
    table_rows = ""
    stats_file = obj_path / f"{object_name}_stats.json"
    
    if stats_file.exists():
        with open(stats_file, "r") as f:
            stats = json.load(f)
        
        # Специальный порядок: сначала методы, потом физика
        important_keys = ["Method I", "Method QU", "Effective Pixels", "R_range_pc"]
        
        for k in important_keys:
            if k in stats:
                v = stats[k]
                val_str = f"{v[0]:.4f} -- {v[1]:.4f}" if isinstance(v, list) else str(v)
                
                # Выделяем аппаратный метод жирным, эвристику - курсивом
                safe_v = val_str.replace('_', r'\_')
                if "Hardware" in safe_v:
                    safe_v = r"\textbf{" + safe_v + "}"
                elif "Heuristic" in safe_v:
                    safe_v = r"\textit{" + safe_v + "}"
                
                safe_k = k.replace('_', r'\_')
                table_rows += f"        {safe_k} & {safe_v} \\\\\n"
        
        # Добавляем остальные параметры из JSON
        # Добавляем остальные параметры из JSON
        for k, v in stats.items():
            if k not in important_keys:
                safe_k = str(k).replace('_', r'\_')
                # Выносим замену слэша в отдельную переменную
                safe_val_content = str(v).replace('_', r'\_') 
                
                val_str = f"{v:.5f}" if isinstance(v, float) else safe_val_content
                
                # Теперь f-строка чистая, без бэкслешей внутри скобок
                table_rows += f"        {safe_k} & {val_str} \\\\\n"
    else:
        table_rows += f"        Distance & {object_dist} pc \\\\\n"
        table_rows += fr"        \textit{{Data Status}} & \color{{red}}\textit{{Stats JSON missing (Heuristic Fallback implied)}} \\" + "\n"
    
    table_rows += f"        Vector Decimation Step & {step} px \\\\\n"

    safe_object_name = object_name.replace('_', r'\_')

    latex_content += f"""\\section*{{Target: {safe_object_name}}}
\\begin{{table}}[H]
    \\centering
    \\caption*{{Physical Parameters and Noise Methodology}}
    \\begin{{tabular}}{{lp{{10cm}}}}
        \\toprule \\textbf{{Parameter}} & \\textbf{{Value}} \\\\ \\midrule
{table_rows}        \\bottomrule
    \\end{{tabular}}
\\end{{table}}

\\begin{{figure}}[H]
    \\centering
    \\begin{{subfigure}}{{0.48\\textwidth}} \\includegraphics[width=\\linewidth]{{../plots/{object_name}/{f_seg}}} 
    \\caption{{Segmentation \& Clump Masking}} \\end{{subfigure}} \\hfill
    \\begin{{subfigure}}{{0.48\\textwidth}} \\includegraphics[width=\\linewidth]{{../plots/{object_name}/{f_snr}}} 
    \\caption{{SNR-Coded B-Field Vectors}} \\end{{subfigure}}
    \\vspace{{0.5cm}}
    \\begin{{subfigure}}{{0.7\\textwidth}} \\includegraphics[width=\\linewidth]{{../plots/{object_name}/{f_sf}}} 
    \\caption{{Structure Function $D_\\phi(R)$}} \\end{{subfigure}}
\\end{{figure}}
\\newpage
"""

latex_content += r"\end{document}"    

with open(REPORT_DIR / "assembled_bistro_report.tex", "w", encoding="utf-8") as f:
    f.write(latex_content)

In [69]:
fits.open("data/BISTRO_Clean/Auriga/maps/Auriga_850um_I_map.fits")[0].header

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                  -64 / number of bits per data pixel                  
NAXIS   =                    3 / number of data axes                            
NAXIS1  =                  271 / length of data axis 1                          
NAXIS2  =                  290 / length of data axis 2                          
NAXIS3  =                    1 / length of data axis 3                          
EXTEND  =                    T / FITS dataset may contain extensions            
COMMENT   FITS (Flexible Image Transport System) format is defined in 'Astronomy
COMMENT   and Astrophysics', volume 376, page 359; bibcode: 2001A&A...376..359H 
LBOUND1 =                 -140 / Pixel origin along axis 1                      
LBOUND2 =                 -152 / Pixel origin along axis 2                      
LBOUND3 =                    1 / Pixel origin along axis 3                      
LABEL   = 'I       '        